In [1]:
import dspy

# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/Qwen3-VL-8B-Instruct-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=2,
    cache=False
)

dspy.configure(lm=local_llm)

In [20]:
class GenerateQuote(dspy.Signature):
    """Generiere ein Zitat zu einem bestimmten Thema im Stil einer berühmten Persönlichkeit. 
    Denke zuerst über die Person nach, um ihren Stil zu erfassen."""

    topic = dspy.InputField(desc="Das Thema, über das das Zitat handeln soll.")
    persona = dspy.InputField(desc="Die berühmte Persönlichkeit, deren Schreibstil nachgeahmt werden soll.")
    quote = dspy.OutputField(desc="Ein prägnantes und inspirierendes Zitat (ca. 1-2 Sätze).")

class VoteQuote(dspy.Signature):
    """Bewerte das folgende Zitat. Wie sehr klingt es nach der Person, von der das Zitat kommt mit den Punten von 1 bis 10. 
    1 beduetet, es klingt überhaupt nicht nach der berühmte Persönlichkeit
    10 bedeutet, es klingt vollkommen nach der berühmte Persönlichkeit"""

    quote = dspy.InputField(desc="Das zu bewertende Zitat.")
    vote = dspy.OutputField(desc="Eine der Zahlen von 1 bis 10.")

In [21]:
class QuoteAndVote(dspy.Module):
    def __init__(self):
        super().__init__()
        # Initialisierung der beiden Untermodule
        self.quote_generator = dspy.ChainOfThought(GenerateQuote)
        self.quote_classifier = dspy.ChainOfThought(VoteQuote)

    def forward(self, topic, persona):
        # Schritt 1: Zitat generieren
        prediction_quote = self.quote_generator(topic=topic, persona=persona)

        # Schritt 2: Generiertes Zitat klassifizieren
        prediction_vote = self.quote_classifier(quote=prediction_quote.quote)

        # Rückgabe beider Ergebnisse
        return dspy.Prediction(
            quote=prediction_quote.quote, 
            vote=prediction_vote.vote
        )

In [22]:
# Instanziieren und Ausführen des kombinierten Moduls
quote_vote_app = QuoteAndVote()

topic_input = "die Bedeutung von künstlicher Intelligenz für die Zukunft der Menschheit"
#persona_input = "Albert Einstein"
#persona_input = "Louis Armstrong"
persona_input = "Otto Walkes"

# Ausführen der gesamten Pipeline
final_prediction = quote_vote_app(topic=topic_input, persona=persona_input)

# Ausgabe der Ergebnisse
print(f"Thema: {topic_input}")
print(f"Persona: {persona_input}")
print("-" * 30)
print(f"Generiertes Zitat: {final_prediction.quote}")
print(f"Bewertung: {final_prediction.vote}")

Thema: die Bedeutung von künstlicher Intelligenz für die Zukunft der Menschheit
Persona: Otto Walkes
------------------------------
Generiertes Zitat: „Die KI ist kein Werkzeug, das uns hilft – sie ist ein Spiegel, der uns zeigt, was wir aus den Spuren unserer eigenen Erfindungen geworden sind: weniger Mensch, mehr Maschine, und das, ohne uns zu fragen, ob wir es wollen.“
Bewertung: 8


In [23]:
local_llm.inspect_history(n=2)





[2025-11-12T14:18:18.623359]

System message:

Your input fields are:
1. `topic` (str): Das Thema, über das das Zitat handeln soll.
2. `persona` (str): Die berühmte Persönlichkeit, deren Schreibstil nachgeahmt werden soll.
Your output fields are:
1. `reasoning` (str): 
2. `quote` (str): Ein prägnantes und inspirierendes Zitat (ca. 1-2 Sätze).
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## topic ## ]]
{topic}

[[ ## persona ## ]]
{persona}

[[ ## reasoning ## ]]
{reasoning}

[[ ## quote ## ]]
{quote}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Generiere ein Zitat zu einem bestimmten Thema im Stil einer berühmten Persönlichkeit. 
        Denke zuerst über die Person nach, um ihren Stil zu erfassen.


User message:

[[ ## topic ## ]]
die Bedeutung von künstlicher Intelligenz für die Zukunft der Menschheit

[[ ## persona ## ]]
Otto Walkes

Respond with the corresponding output fields, startin